<!-- applied-genai-header -->
<div style="background: linear-gradient(135deg, #1e4b8f 0%, #2d6cb8 100%); color: white; padding: 24px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; margin-bottom: 16px;">
  <div style="font-size: 12px; opacity: 0.85; letter-spacing: 1.5px; text-transform: uppercase;">Applied Generative AI · IE 5373</div>
  <h1 style="margin: 8px 0 4px 0; font-size: 28px; font-weight: 600;">Prompting with the OpenAI API</h1>
  <div style="font-size: 14px; opacity: 0.9;">First steps: chat completions, system/user/assistant roles, parameters</div>
  <div style="margin-top: 12px; font-size: 12px; opacity: 0.8;">Prof. Mohammad Dehghani · Northeastern University</div>
</div>

> **📌 Note on models.** This lab references specific LLM versions (e.g. `gpt-5`, `gpt-5-mini`).
> Models update quickly — you are welcome (and encouraged) to swap in any newer OpenAI / Anthropic / Google model you have access to.
> The default model is set in one place: `DEFAULT_CHAT_MODEL` inside `utils.py`. Change it there and every cell follows.


In [5]:
!pip install --upgrade openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 25.1 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0


In [2]:
# === Shared lab setup: utils.py + API key + sticky lab pill ===
# Downloads the shared utilities (pretty_print, model constants, key loader,
# lab_pill) from the AppliedGenAI repo so every notebook stays small and
# consistent. The API key is read from a Colab secret named OPENAI_API_KEY
# (set it once under Colab → 🔑 → "Notebook access" — same name in every lab).
import os
if not os.path.exists("utils.py"):
    !wget -q https://raw.githubusercontent.com/mdehghani86/AppliedGenAI/main/utils.py -O utils.py

from utils import (
    pretty_print,
    DEFAULT_CHAT_MODEL,   # e.g. "gpt-5"  — main reasoning model
    DEFAULT_MINI_MODEL,   # e.g. "gpt-5-mini"  — cheaper / faster default
    DEFAULT_EMBED_MODEL,  # e.g. "text-embedding-3-small"
    get_openai_key,
    lab_pill,
)

lab_pill('Prompting with the OpenAI API')        # sticky banner so you always see which lab you're in
get_openai_key(verify=True);    # loads the key + pings OpenAI to confirm it works


⚬	**ImportError:** An outdated openai==0.28 package lacked the required OpenAI class. Fix: Upgraded the package and restarted the runtime.

⚬	**Exposed API Key:** get_openai_key(verify=True) auto-displayed the secret key because it was the final line in the Colab cell. Fix: Added a trailing semicolon (;) to suppress the output.

⚬	**Takeaway:** Always check package versions and clear cell outputs before saving notebooks containing credentials.

# Applied Generative AI
Instructor: Prof. Dehghani
Welcome to the Applied Generative AI course. In this course, we will explore the foundations and applications of Generative AI using tools like OpenAI's API. By the end of this session, you will:

🟢 Understand how to set up and connect to OpenAI's API.
🌟 Learn about the roles (System, Assistant, User) in prompt design.
✨ Generate text, images, and vector embeddings programmatically.
🔧 Explore fine-tuning to customize AI models for specific tasks.
Let’s get started with setting up the OpenAI API!

In [1]:
# Install the OpenAI Python SDK
# This library allows us to interact with OpenAI's API for text, images, and embeddings.
# !pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [3]:
from google.colab import userdata
import openai

# Retrieve the key from Colab secrets
# (api key handled by get_openai_key() in setup cell)

# Confirm the key was loaded (for debugging only; don’t print your real key in shared notebooks!)
print("Key loaded:", "Yes" if os.environ.get("OPENAI_API_KEY") else "No")


Key loaded: Yes


🎯 Prompt Playground: Understanding Roles
In OpenAI's API, you interact with the model using roles, which define the flow of the conversation:

➡️ System: Sets the behavior and tone of the assistant (e.g., "You are a cheerful assistant.").
➡️ User: Represents the input or question from the user (e.g., "What is AI?").
➡️ Assistant: Automatically generated responses based on the system and user inputs.
💡 Why Roles Matter: Roles help control the assistant's personality and the quality of responses. For example:

A system message like "You are a strict teacher" makes the assistant respond more formally.
A system message like "You are a friendly chatbot" leads to casual responses.
Let’s see how these roles work in the next cell!

## Example 1: Without Assistant Role

In [4]:
from openai import OpenAI
client = OpenAI()

import openai

response = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

pretty_print(response.choices[0].message.content, title="🤖 Model Response")


# Example 2: With Assistant Role

In [5]:
from openai import OpenAI
client = OpenAI()

# Demonstrating roles in OpenAI's API with the assistant role

# Define the conversation including a predefined assistant response
messages = [
    {"role": "system", "content": "You are a math tutor who explains problems step by step."},  # System role sets the behavior
    {"role": "user", "content": "Solve for x: 2x + 5 = 15"},  # User question
    {"role": "assistant", "content": "To solve for x: \n1. Subtract 5 from both sides: 2x = 10\n2. Divide both sides by 2: x = 5"}  # Predefined assistant response
]

# Send the conversation to the API
response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,  # Use the chosen model
    messages=messages  # Pass the conversation
)

# Print the assistant's response
pretty_print(response.choices[0].message.content, title="Assistant's Response:")

In Example 1, using only system and user roles, the model's response was simple.
In Example 2, I added a predefined assistant message before the conversation continued; the response was more detail in comparing.This suggests that including a sample assistant turn in the problem solving processs.

🧮 Hands-On

In [ ]:
from openai import OpenAI
client = OpenAI()

# ++++ Hands-On: Complete the Roles ++++

# Define the conversation using the role structure
messages = [
    {"role": "system", "content": "You are a math tutor who explains concepts clearly and step by step."},  # Complete the system role
    {"role": "user", "content": "How do you calculate the arithmetic mean of a set of numbers?"},  # Complete the user role
    {"role": "assistant", "content": "To calculate the arithmetic mean:\n"
                                "1. Add all the numbers in the set.\n"
                                "2. Divide the sum by the number of numbers.\n\n"
                                "For example, for 10, 20, and 30:\n"
                                "Mean = (10 + 20 + 30) / 3 = 60 / 3 = 20."
                                }  # Optional predefined assistant response
]

# Uncomment the code below after completing the placeholders
response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,  # Specify the model
    messages=messages  # Pass the completed conversation
)
pretty_print(response.choices[0].message.content, title="Assistant's Response:")

Exercise 1: Simple Q&A with System + User Roles
👉 Task:
Write a prompt that asks the assistant to behave like a science teacher, then ask it a science-related question.

In [ ]:
from openai import OpenAI
client = OpenAI()

# 🧠 Define your own roles and prompt below
messages = [
    {"role": "------", "content": "You are a [insert role/personality, e.g., 'science teacher', 'debate coach']."},
    {"role": "------", "content": "Ask your question here, e.g., 'Explain how photosynthesis works.'"}
]

# 🧠 Call the OpenAI API with your customized prompt
response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,
    messages=messages
)

# 🧠 Print the model's response
pretty_print(response.choices[0].message.content, title="Assistant's Response:")

✍️ Exercise 2: Give a prompt of your choice.
👉 Task:
Now write your own custom prompt. Use any role, any question — be creative!

In [ ]:
from openai import OpenAI
client = OpenAI()

# Replace with your own prompt idea!
messages = [
    {"role": "system", "content": "You are a ____."},
    {"role": "user", "content": "____?"}
]

response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,
    messages=messages
)

pretty_print(response.choices[0].message.content, title="Assistant's Response:")
